# Outlier Strategy Comparison

Compare interval-local legacy filtering with shot-wide local statistical filtering for one AUG shot and interval. Both branches use the same full-shot mapping and fitting settings; only the statistical outlier scope differs.

In [ ]:
import copy
import sys
from contextlib import redirect_stdout
from io import StringIO
from pathlib import Path
import importlib
import pandas as pd
import numpy as np
import yaml
from IPython.display import display

current_path = Path.cwd().resolve()
search_roots = [current_path, *current_path.parents, current_path / 'TS_profile_fitter']
project_root = next(
    (path for path in search_roots if (path / 'src' / 'ts_profile_fitter').is_dir()),
    None,
)
if project_root is None:
    raise FileNotFoundError('Could not locate the TS_profile_fitter project root.')
sys.path.insert(0, str(project_root / 'src'))
sys.path.insert(0, str(project_root.parent / 'aug_separatrix'))
import get_connection_length_selectable as aug_connection_length
importlib.reload(aug_connection_length)
get_LcSOL = aug_connection_length.get_LcSOL

from ipfnpytools.plot import use_base_style
from ts_profile_fitter.loading import load_intervals, load_ts_data
from ts_profile_fitter.plotting import plot_profile_fit
import ts_profile_fitter.preprocess as preprocess
importlib.reload(preprocess)
import ts_profile_fitter.pipeline as pipeline
importlib.reload(pipeline)
from ts_profile_fitter.pipeline import run_interval
from ts_profile_fitter.preprocess import preprocess_ts_data

use_base_style()
new_config_path = project_root / 'configs' / 'pipeline_config_run05_new_outlier.yaml'
legacy_config_path = project_root / 'configs' / 'config_with_inner_core.yaml'
with new_config_path.open('r', encoding='utf-8') as config_file:
    new_pipeline_config = yaml.safe_load(config_file) or {}
with legacy_config_path.open('r', encoding='utf-8') as config_file:
    legacy_pipeline_config = yaml.safe_load(config_file) or {}


def resolve_config_path(config, key, default):
    path = Path(config.get(key, default))
    return path if path.is_absolute() else new_config_path.parent / path


new_fit_config = copy.deepcopy(new_pipeline_config['fit_config'])
legacy_fit_config = copy.deepcopy(legacy_pipeline_config['fit_config'])
new_preprocess_config = new_fit_config['preprocess']
legacy_preprocess_config = legacy_fit_config['preprocess']
local_statistical_config = new_preprocess_config.get('local_statistical_outlier_config')
if not local_statistical_config or not local_statistical_config.get('enabled', True):
    raise ValueError('The new-outlier configuration must enable local statistical filtering.')

# Use one shared mapping so the comparison isolates the configured filters.
mapping_method = legacy_preprocess_config.get('mapping_method', 'ipfnpytools')
r_edge_offset = legacy_preprocess_config.get('r_edge_offset', 0.008)
data_directory = resolve_config_path(new_pipeline_config, 'data_dir', '../data')
intervals_path = resolve_config_path(new_pipeline_config, 'database', '../data/intervals.csv')
equilibrium = legacy_pipeline_config.get('equilibrium', 'EQH')
diagnostic = legacy_pipeline_config.get('diagnostic', 'VTA')

shot_id = 35970
time_requested = 4.4
output_directory = project_root / 'plots'
save_figures = False
show_figures = True
close_figures = False

In [ ]:
intervals_df = load_intervals(str(intervals_path))
matching_intervals = intervals_df.loc[
    intervals_df['shot_id'].eq(shot_id)
    & intervals_df['t_begin'].le(time_requested)
    & intervals_df['t_end'].ge(time_requested)
]
if matching_intervals.empty:
    raise ValueError(
        f'No interval found for shot {shot_id} at t={time_requested:.3f} s.'
    )

selected_interval = matching_intervals.iloc[0]
interval = {
    't_begin': float(selected_interval['t_begin']),
    't_end': float(selected_interval['t_end']),
    'label': selected_interval['label'],
}
interval_begin = interval['t_begin']
interval_end = interval['t_end']

ts_data = load_ts_data(
    shot_id,
    str(data_directory),
    diagnostic=diagnostic,
    equilibrium=equilibrium,
)

mapped_shot = preprocess_ts_data(
    ts_data,
    interval=None,
    r_edge_offset=r_edge_offset,
    mapping_method=mapping_method,
    apply_filters=False,
)
mapped_shot['_comparison_row_id'] = range(len(mapped_shot))

print(
    f'Shot {shot_id} at {time_requested:.3f} s -> '
    f'interval [{interval_begin:.3f}, {interval_end:.3f}] s ({interval["label"]})'
)
print(f'Mapped full-shot measurements: {len(mapped_shot):,}')

In [ ]:
new_fit_config['fits']

In [ ]:
#new_fit_config['fits']['spline']['knot_strategy'] = 'balanced_quantile'
#new_fit_config['fits']['spline']['knot_count_shift'] = 0
#new_fit_config.setdefault('physics', {}).setdefault('connection_length', {})['plot'] = True

#new_fit_config['fits']['spline']['knot_strategy'], new_fit_config['fits']['spline']['knot_count_shift']

In [ ]:
pwd

In [ ]:
legacy_diagnostics = {}
legacy_filter_log = StringIO()
with redirect_stdout(legacy_filter_log):
    legacy_result = run_interval(
        shot_id,
        interval,
        ts_data,
        fit_config=legacy_fit_config,
        make_plots=False,
        diagnostic_context=legacy_diagnostics,
        mapped_shot_data=mapped_shot,
    )

new_diagnostics = {}
new_filter_log = StringIO()
with redirect_stdout(new_filter_log):
    new_result = run_interval(
        shot_id,
        interval,
        ts_data,
        fit_config=new_fit_config,
        make_plots=False,
        diagnostic_context=new_diagnostics,
        mapped_shot_data=mapped_shot,
        connection_length_provider=get_LcSOL,
    )

print('Both YAML-configured filtering and fitting runs completed.')

In [ ]:
insufficient_support_rows = new_diagnostics[
    'local_statistical_outlier_metadata'
].get('insufficient_support_rows')
if insufficient_support_rows is None:
    raise RuntimeError('The statistical filter did not return support diagnostics.')

if insufficient_support_rows.empty:
    print('No measurements in this interval were skipped for insufficient support.')
else:
    support_summary = (
        insufficient_support_rows
        .groupby(['system', 'quantity', 'metric', 'reason'], dropna=False)
        .agg(
            skipped_checks=('row_index', 'size'),
            distinct_points=('row_index', 'nunique'),
            min_valid_neighbors=('neighbor_count', 'min'),
            max_valid_neighbors=('neighbor_count', 'max'),
            min_radial_support=('radial_support', 'min'),
            min_temporal_support=('temporal_support', 'min'),
        )
        .reset_index()
    )
    display(support_summary)
    display(insufficient_support_rows.sort_values(
        ['time', 'system', 'channel_index', 'quantity', 'metric']
    ).reset_index(drop=True))

In [ ]:
import ts_profile_fitter.plotting as plotting
importlib.reload(plotting)
plot_profile_fit = plotting.plot_profile_fit
raw_interval = mapped_shot.loc[
    mapped_shot['time'].between(interval_begin, interval_end)
].copy()


def rejected_interval_rows(result):
    retained = result['_mapped']
    retained_ids = retained['_comparison_row_id'].astype(int)
    rejected = raw_interval.loc[
        ~raw_interval['_comparison_row_id'].isin(retained_ids)
    ].copy()
    if len(retained) + len(rejected) != len(raw_interval):
        raise RuntimeError('Retained and rejected rows do not partition the interval input.')
    return rejected


legacy_rejected_rows = rejected_interval_rows(legacy_result)
new_rejected_rows = rejected_interval_rows(new_result)
profile_runs = {
    'Legacy: config_with_inner_core': {
        'result': legacy_result,
        'rejected': legacy_rejected_rows,
    },
    'Shot-wide: run05 statistical': {
        'result': new_result,
        'rejected': new_rejected_rows,
    },
}

comparison_summary = pd.DataFrame([
    {
        'strategy': strategy_name,
        'interval input points': len(raw_interval),
        'removed points': len(strategy_data['rejected']),
        'removed core points': int(strategy_data['rejected']['system'].eq('core').sum()),
        'removed edge points': int(strategy_data['rejected']['system'].eq('edge').sum()),
        'retained points': len(strategy_data['result']['_mapped']),
    }
    for strategy_name, strategy_data in profile_runs.items()
]).set_index('strategy')
display(comparison_summary)

In [ ]:
from ts_profile_fitter.plotting import (
    plot_filtered_profile_with_outliers,
    PROFILE_RADIUS_LABEL,
)

profile_runs['Legacy: config_with_inner_core']['diagnostics'] = legacy_diagnostics
profile_runs['Shot-wide: run05 statistical']['diagnostics'] = new_diagnostics
origin_strategy_name = 'Shot-wide: run05 statistical'

## Rejected Measurements by Reason

Counts and point-level values are listed for each recorded filter reason. Filter `outliers_by_reason` by `outlier_reason` to inspect a family with the existing profile plot functions.

In [ ]:
reasoned_frames = []
preprocess_removed = new_diagnostics.get('preprocess_removed_rows')
if preprocess_removed is not None and not preprocess_removed.empty:
    reasoned_frames.append(preprocess_removed)

for metadata_key in (
    'local_statistical_outlier_metadata',
    'density_outside_separatrix_metadata',
    'cold_dense_edge_metadata',
    'distance_outlier_metadata',
):
    stage_metadata = new_diagnostics.get(metadata_key)
    if stage_metadata is not None:
        stage_removed = stage_metadata.get('removed_rows')
        if stage_removed is not None and not stage_removed.empty:
            reasoned_frames.append(stage_removed)

if reasoned_frames:
    reasoned_outliers = pd.concat(reasoned_frames, ignore_index=True)
else:
    reasoned_outliers = new_rejected_rows.head(0).copy()
    reasoned_outliers['filter_reason'] = pd.Series(dtype=str)

rejected_ids = set(new_rejected_rows['_comparison_row_id'])
reasoned_outliers = reasoned_outliers.loc[
    reasoned_outliers['_comparison_row_id'].isin(rejected_ids)
].drop_duplicates(['_comparison_row_id', 'filter_reason'])
reasoned_outliers['outlier_reason'] = (
    reasoned_outliers['filter_reason'].fillna('').str.split(';'))
outliers_by_reason = reasoned_outliers.explode('outlier_reason')
outliers_by_reason['outlier_reason'] = (
    outliers_by_reason['outlier_reason'].astype(str).str.strip())
outliers_by_reason = outliers_by_reason.loc[
    outliers_by_reason['outlier_reason'].ne('')
].copy()

reasoned_ids = set(outliers_by_reason['_comparison_row_id'])
missing_reason_ids = rejected_ids - reasoned_ids
unreasoned_outliers = new_rejected_rows.loc[
    new_rejected_rows['_comparison_row_id'].isin(missing_reason_ids)
].copy()

reason_summary = outliers_by_reason.groupby('outlier_reason').agg(
    points=('_comparison_row_id', 'nunique'),
    core=('system', lambda values: values.eq('core').sum()),
    edge=('system', lambda values: values.eq('edge').sum()),
).sort_values('points', ascending=False)
display(reason_summary)

reason_detail_columns = [
    column for column in (
        '_comparison_row_id', 'shot', 'system', 'channel_index', 'time',
        'r_midplane', 'Te', 'Te_err', 'Ne', 'Ne_err', 'outlier_reason',
    )
    if column in outliers_by_reason.columns
]
display(
    outliers_by_reason[reason_detail_columns]
    .sort_values(['outlier_reason', 'time', 'system', 'channel_index'])
    .reset_index(drop=True)
)

if not unreasoned_outliers.empty:
    print(
        f'{len(unreasoned_outliers)} rejected measurements have no recorded '
        'filter reason in the captured stage metadata:'
    )
    unreasoned_columns = [
        column for column in (
            '_comparison_row_id', 'shot', 'system', 'channel_index', 'time',
            'r_midplane', 'rho_pol', 'Te', 'Te_err', 'Ne', 'Ne_err',
        )
        if column in unreasoned_outliers.columns
    ]
    display(
        unreasoned_outliers[unreasoned_columns]
        .sort_values(['time', 'system', 'channel_index'])
        .reset_index(drop=True)
    )

In [ ]:
reason_tags_by_row = reasoned_outliers.groupby('_comparison_row_id')['filter_reason'].agg(
    lambda values: '; '.join(dict.fromkeys(
        tag.strip()
        for value in values.dropna().astype(str)
        for tag in value.split(';')
        if tag.strip()
    ))
)
outliers_with_reasons = new_rejected_rows.merge(
    reason_tags_by_row.rename('filter_reason'),
    how='left',
    left_on='_comparison_row_id',
    right_index=True,
)
outliers_with_reasons['filter_reason'] = (
    outliers_with_reasons['filter_reason'].fillna(''))

## Filtered Profile and Outliers

Filtered measurements are colored by time. Outlier markers are grouped as `Negative or zero`, `core channel removal`, `Other`, `Te_stat`, and `Ne_stat` by their recorded `filter_reason`.

In [ ]:
plot_filtered_profile_with_outliers(
    filtered=new_result['_mapped'],
    outliers=outliers_with_reasons,
    quantity=('Te', 'Ne'),
    #directory=str(output_directory),
    save_fig=False,
    directory = '/shares/users/work/joanune/Desktop/TESE_Joao_Pedro_Nunes/chapters/Formulation_and_Methods/Data_Processing/figures',
    show=show_figures,
    close=close_figures,
    nshot=shot_id,
    interval=interval,
    equilibrium=equilibrium,
    fig_name=f'{shot_id}_shot_wide_te_ne_filtered_outliers',
    y_mins_boxes=[-30, -5e18],
    y_maxs_boxes=[1000, 0.8e20],
    x_mins=2.12,
    x_maxs=2.25,
    h_factor=1.5,
    v_factor=2
)

In [ ]:
radius_window = (2.22, 2.24)
#radius_window = (2.18,2.2)
nearby_Te_flags = outliers_with_reasons.loc[
    outliers_with_reasons['r_midplane'].between(*radius_window)
    & outliers_with_reasons['filter_reason'].str.contains(
        'Te_statistical_', na=False)
].copy()
nearby_Te_flags['Te_relative_error'] = (
    nearby_Te_flags['Te_err'] / nearby_Te_flags['Te'].abs())
nearby_Te_columns = [
    column for column in (
        '_comparison_row_id', 'system', 'channel_index', 'time', 'r_midplane',
        'Te', 'Te_err', 'Te_relative_error', 'filter_reason',
    )
    if column in nearby_Te_flags.columns
]
display(
    nearby_Te_flags[nearby_Te_columns]
    .sort_values(['r_midplane', 'time'])
    .reset_index(drop=True)
)

from ts_profile_fitter.preprocess import (
    _apply_channel_trim,
    _apply_measurement_validity_filters,
    _apply_rhop_filter,
)

active_preprocess_config = new_fit_config['preprocess']
local_filter_config = active_preprocess_config['local_statistical_outlier_config']
default_threshold = float(local_filter_config.get('threshold', 3.5))
upper_value_threshold = float(
    local_filter_config.get('upper_value_threshold', default_threshold))
relative_error_threshold = float(
    local_filter_config.get('relative_error_threshold', default_threshold))
support_data, _ = _apply_measurement_validity_filters(
    mapped_shot, return_metadata=True)
support_data = _apply_rhop_filter(
    support_data, rhop_filter=active_preprocess_config.get('rhop_filter'))
support_data = _apply_channel_trim(
    support_data, channel_trim=active_preprocess_config.get('channel_trim'))


def _neighborhood_cutoff(values, threshold):
    values = pd.to_numeric(values, errors='coerce').to_numpy(dtype=float)
    values = values[np.isfinite(values)]
    if values.size == 0:
        return 0, np.nan, np.nan, np.nan
    median = float(np.median(values))
    mad = float(np.median(np.abs(values - median)))
    cutoff = median + threshold * 1.4826 * mad
    return int(values.size), median, mad, float(cutoff)


neighborhood_summaries = []
neighborhood_rows = []
for _, target in nearby_Te_flags.iterrows():
    target_id = int(target['_comparison_row_id'])
    system_data = support_data.loc[
        support_data['system'].eq(target['system'])]
    radial_width = local_filter_config['radial_half_width_m'][target['system']]
    time_width = float(local_filter_config['time_half_width_s'])
    radial_delta = np.abs(system_data['r_midplane'] - target['r_midplane'])
    time_delta = np.abs(system_data['time'] - target['time'])
    neighbors = system_data.loc[
        radial_delta.le(radial_width)
        & time_delta.le(time_width)
        & system_data['_comparison_row_id'].ne(target_id)
    ].copy()

    if 'channel_index' in support_data.columns and pd.notna(target['channel_index']):
        spatial_channels = set(neighbors['channel_index'].dropna())
        spatial_channels.add(target['channel_index'])
        required_channels = max(
            int(local_filter_config.get('min_radial_neighbors', 2)), 2)
        channels_to_add = max(0, required_channels - len(spatial_channels))
        if channels_to_add:
            radial_column = (
                'rho_pol' if 'rho_pol' in support_data.columns else 'r_midplane')
            temporal = system_data.loc[time_delta.le(time_width)]
            candidates = temporal.loc[
                temporal['channel_index'].notna()
                & ~temporal['channel_index'].isin(spatial_channels)
            ].copy()
            candidates['_radial_distance'] = np.abs(
                candidates[radial_column] - target[radial_column])
            channel_distances = candidates.groupby(
                'channel_index')['_radial_distance'].median()
            nearest_channels = channel_distances.nsmallest(channels_to_add).index
            fallback_rows = temporal.loc[
                temporal['channel_index'].isin(nearest_channels)]
            neighbors = pd.concat([neighbors, fallback_rows]).drop_duplicates(
                '_comparison_row_id')

    neighbor_density = neighbors['Te']
    neighbor_relative_error = (
        neighbors['Te_err'] / neighbors['Te'].abs())
    n_density, density_median, density_mad, density_cutoff = (
        _neighborhood_cutoff(neighbor_density, upper_value_threshold))
    n_relerr, relerr_median, relerr_mad, relerr_cutoff = (
        _neighborhood_cutoff(
            neighbor_relative_error, relative_error_threshold))
    neighborhood_summaries.append({
        '_comparison_row_id': target_id,
        'time': target['time'],
        'r_midplane': target['r_midplane'],
        'Te': target['Te'],
        'Te_relative_error': target['Te_relative_error'],
        'neighbor_count': n_relerr,
        'neighbor_channels': neighbors['channel_index'].nunique(),
        'Te_neighbor_median': density_median,
        'Te_neighbor_MAD': density_mad,
        'Te_upper_value_cutoff': density_cutoff,
        'relative_error_neighbor_median': relerr_median,
        'relative_error_neighbor_MAD': relerr_mad,
        'relative_error_cutoff': relerr_cutoff,
        'reason': target['filter_reason'],
    })
    neighbor_view = neighbors.assign(
        _target_row_id=target_id,
        Te_relative_error=neighbor_relative_error,
        rejected_in_interval=neighbors['_comparison_row_id'].isin(
            set(new_rejected_rows['_comparison_row_id'])),
    )
    neighborhood_rows.append(neighbor_view[[
        '_target_row_id', '_comparison_row_id', 'channel_index', 'time',
        'r_midplane', 'Te', 'Te_err', 'Te_relative_error',
        'rejected_in_interval',
    ]])

display(pd.DataFrame(neighborhood_summaries))

In [ ]:
neighbors['channel_index']

In [ ]:
retained_high_te = new_result['_mapped'].loc[
    new_result['_mapped']['system'].eq('edge')
    & new_result['_mapped']['r_midplane'].between(*radius_window)
    & new_result['_mapped']['Te'].gt(600)
].copy()

candidate_cutoffs = []
for _, target in retained_high_te.iterrows():
    target_id = target['_comparison_row_id']
    candidate_support = support_data.loc[
        support_data['system'].eq(target['system'])
    ].copy()
    if 'shot' in support_data.columns and pd.notna(target.get('shot')):
        candidate_support = candidate_support.loc[
            candidate_support['shot'].eq(target['shot'])]

    radial_width = local_filter_config['radial_half_width_m'][target['system']]
    time_width = float(local_filter_config['time_half_width_s'])
    radial_delta = np.abs(
        candidate_support['r_midplane'] - target['r_midplane'])
    time_delta = np.abs(candidate_support['time'] - target['time'])
    neighbors = candidate_support.loc[
        radial_delta.le(radial_width)
        & time_delta.le(time_width)
        & candidate_support['_comparison_row_id'].ne(target_id)
    ].copy()

    if 'channel_index' in support_data.columns and pd.notna(target['channel_index']):
        spatial_channels = set(neighbors['channel_index'].dropna())
        spatial_channels.add(target['channel_index'])
        required_channels = max(
            int(local_filter_config.get('min_radial_neighbors', 2)), 2)
        channels_to_add = max(0, required_channels - len(spatial_channels))
        if channels_to_add:
            radial_column = (
                'rho_pol' if 'rho_pol' in support_data.columns else 'r_midplane')
            temporal = candidate_support.loc[time_delta.le(time_width)]
            fallback_candidates = temporal.loc[
                temporal['channel_index'].notna()
                & ~temporal['channel_index'].isin(spatial_channels)
            ].copy()
            fallback_candidates['_radial_distance'] = np.abs(
                fallback_candidates[radial_column] - target[radial_column])
            channel_distances = fallback_candidates.groupby(
                'channel_index')['_radial_distance'].median()
            nearest_channels = channel_distances.nsmallest(channels_to_add).index
            fallback_rows = temporal.loc[
                temporal['channel_index'].isin(nearest_channels)]
            neighbors = pd.concat([neighbors, fallback_rows]).drop_duplicates(
                '_comparison_row_id')

    neighbor_values = pd.to_numeric(neighbors['Te'], errors='coerce')
    valid_neighbors = neighbors.loc[np.isfinite(neighbor_values)].copy()
    values = neighbor_values.loc[valid_neighbors.index].to_numpy(dtype=float)
    median = float(np.median(values)) if values.size else np.nan
    mad = float(np.median(np.abs(values - median))) if values.size else np.nan
    cutoff = median + upper_value_threshold * 1.4826 * mad
    candidate_cutoffs.append({
        '_comparison_row_id': target_id,
        'time': target['time'],
        'r_midplane': target['r_midplane'],
        'Te': target['Te'],
        'Te_err': target['Te_err'],
        'neighbor_count': len(valid_neighbors),
        'neighbor_channels': valid_neighbors['channel_index'].nunique(),
        'neighbor_median': median,
        'neighbor_MAD': mad,
        'upper_value_threshold': upper_value_threshold,
        'computed_Te_cutoff': cutoff,
        'exceeds_cutoff': bool(target['Te'] > cutoff),
    })

display(pd.DataFrame(candidate_cutoffs))

In [ ]:
candidate_neighbor_detail = neighbors.assign(
    Te_relative_error=neighbors['Te_err'] / neighbors['Te'].abs(),
    rejected_in_interval=neighbors['_comparison_row_id'].isin(
        set(new_rejected_rows['_comparison_row_id'])),
)[[
    '_comparison_row_id', 'channel_index', 'time', 'r_midplane',
    'Te', 'Te_err', 'Te_relative_error', 'rejected_in_interval',
]].sort_values(['Te', 'time'], ascending=[False, True])
display(candidate_neighbor_detail)

## Fitted Profiles Without Outlier Overlays

The first fit-quality plot shows the filtered Te and Ne profiles with modified-tanh fits. The following plots show the corresponding monotonic spline fits, with knot locations marked below the measurements.

In [ ]:
import importlib
import ts_profile_fitter.plotting as plotting
from ts_profile_fitter.figure_naming import make_fig_name
plotting = importlib.reload(plotting)
plot_filtered_profile_with_outliers = plotting.plot_filtered_profile_with_outliers

plot_filtered_profile_with_outliers(
    filtered=new_result['_mapped'],
    outliers=None,
    quantity=('Te', 'Ne'),
    directory='/shares/users/work/joanune/Desktop/TESE_Joao_Pedro_Nunes/chapters/Formulation_and_Methods/Profile_Fitting/figures',
    save_fig=False,
    show=show_figures,
    close=close_figures,
    nshot=shot_id,
    interval=interval,
    equilibrium=equilibrium,
    fig_name = make_fig_name(shot_id, equilibrium, interval = interval, n_knots = 0,plot_kind='wide_te_ne_fit_quality'),
    popt={
        quantity: new_result['mtanh'][quantity]['params']
        for quantity in ('Te', 'Ne')
    },
    fit_metrics=new_result['fit_metrics']['mtanh'],
    fit_metrics_loc='center right',
    show_fit=True,
    show_outliers=False,
    h_factor=1.5,
    v_factor=2.0,
    legend_kw={'loc': 'best'}
)

In [ ]:
import importlib
plotting = importlib.reload(plotting)

spline_plot_data = new_result['_mapped']
spline_core = spline_plot_data.loc[spline_plot_data['system'].eq('core')]
spline_edge = spline_plot_data.loc[spline_plot_data['system'].eq('edge')]

plotting.plot_spline_profile_fit(
    x_core=spline_core['r_midplane'].to_numpy(),
    x_edge=spline_edge['r_midplane'].to_numpy(),
    y_core={
        quantity: spline_core[quantity].to_numpy()
        for quantity in ('Te', 'Ne')
    },
    y_edge={
        quantity: spline_edge[quantity].to_numpy()
        for quantity in ('Te', 'Ne')
    },
    y_err_core={
        quantity: spline_core[f'{quantity}_err'].to_numpy()
        for quantity in ('Te', 'Ne')
    },
    y_err_edge={
        quantity: spline_edge[f'{quantity}_err'].to_numpy()
        for quantity in ('Te', 'Ne')
    },
    ds_out=new_result['_ds_out'],
    quantity=('Te', 'Ne'),
    x_labels=PROFILE_RADIUS_LABEL,
    y_labels=['Te [eV]', 'Ne [m$^{-3}$]'],
    knot_pos=new_result['_knots'],
    time_c=spline_core['time'].to_numpy(),
    time_e=spline_edge['time'].to_numpy(),
    directory='/shares/users/work/joanune/Desktop/TESE_Joao_Pedro_Nunes/chapters/Formulation_and_Methods/Profile_Fitting/figures',
    close=close_figures,
    save_fig=False,
    show=show_figures,
    nshot=shot_id,
    interval=interval,
    equilibrium=equilibrium,
    fig_name=f'{shot_id}_shot_wide_te_ne_spline_fit_quality',
    fit_metrics=new_result['fit_metrics']['spline'],
    fit_metrics_loc='center right',
    h_factor=1.5,
    v_factor=2.0,
    legend_kw={'loc': 'best'},
)

In [ ]:
plots[]

In [ ]:
import numpy as np

mapped = new_result['_mapped']
core = mapped.loc[mapped['system'].eq('core')]
edge = mapped.loc[mapped['system'].eq('edge')]
ds_out = new_result['_ds_out']
physics_result = new_result['physics']
spline_physics = physics_result['spline']
mtanh_physics = physics_result['mtanh']

core_r = core['r_midplane'].to_numpy(dtype=float)
edge_r = edge['r_midplane'].to_numpy(dtype=float)
core_te = core['Te'].to_numpy(dtype=float)
edge_te = edge['Te'].to_numpy(dtype=float)
core_te_err = core['Te_err'].to_numpy(dtype=float)
edge_te_err = edge['Te_err'].to_numpy(dtype=float)

measured_r = np.concatenate([core_r, edge_r])
measured_r = measured_r[np.isfinite(measured_r)]
core_channels = np.sort(np.unique(core_r[np.isfinite(core_r)]))
#r_zoom_min = core_channels[-2] if core_channels.size >= 2 else np.nanmin(measured_r)
r_zoom_min=2.10
#r_zoom_max = np.nanmax(measured_r)
r_zoom_max=2.20

te_fit_r = np.linspace(np.nanmin(measured_r), np.nanmax(measured_r), 500)
te_mtanh = plotting.modified_tanh(
    te_fit_r, *np.asarray(new_result['mtanh']['Te']['params'], dtype=float)
)
spline_r = ds_out['R_mid_spl'].to_numpy(dtype=float)
te_spline = ds_out['Te_spl'].to_numpy(dtype=float)
te_measurements = np.concatenate([core_te, edge_te])
te_measurements = te_measurements[np.isfinite(te_measurements)]
knot_y = np.nanmin(te_measurements) - 0.04 * (np.ptp(te_measurements) or 1.0)
knots = np.asarray(new_result['_knots'], dtype=float)

spline_r_physics = np.asarray(spline_physics['R_grid'], dtype=float)
mtanh_r_physics = np.asarray(mtanh_physics['R_grid'], dtype=float)
spline_lambda = 1e3 * np.asarray(spline_physics['lam_Te_grid'], dtype=float)
mtanh_lambda = 1e3 * np.asarray(mtanh_physics['lam_Te_grid'], dtype=float)

x_data = [
    core_r, edge_r, core_r, edge_r, spline_r, te_fit_r, knots,
    spline_r_physics, mtanh_r_physics,
    spline_r_physics, spline_r_physics, mtanh_r_physics, mtanh_r_physics,
]
y_data = [
    core_te, edge_te, core_te, edge_te, te_spline, te_mtanh,
    np.full(knots.size, knot_y), spline_lambda, mtanh_lambda,
    1e-9 * np.asarray(spline_physics['q_par_P_SOL_grid'], dtype=float),
    1e-9 * np.asarray(spline_physics['q_par_SH_grid'], dtype=float),
    1e-9 * np.asarray(mtanh_physics['q_par_P_SOL_grid'], dtype=float),
    1e-9 * np.asarray(mtanh_physics['q_par_SH_grid'], dtype=float),
]
ey_data = [
    core_te_err, edge_te_err, None, None, None, None, None,
    None, None, None, None, None, None,
]
boxes = [0] * 7 + [1] * 2 + [2] * 4

labels = [
    None, None, 'Core TS', 'Edge TS', 'Spline', 'mTanh', 'Knots',
    r'$\lambda_{Te,\mathrm{spline}}$', r'$\lambda_{Te,\mathrm{mTanh}}$',
    r'$q_{e,\parallel,\mathrm{SOL,spline}}$', r'$q_{e,\parallel,\mathrm{cond,spline}}$',
    r'$q_{e,\parallel,\mathrm{SOL,mtanh}}$', r'$q_{e,\parallel,\mathrm{cond,mtanh}}$',
]
linestyles = ['', '', '', '', '-', '-', '', '-', '-', '-', '-', '--', '--']
markers = [None, None, 'x', 'v', None, None, 'o'] + [None] * 6
markersizes = [0, 0, 6, 6, None, None, 4] + [None] * 6
plot_kw = [
    {'color': 'tab:blue', 'ecolor': 'tab:blue', 'alpha': 0.45},
    {'color': 'tab:orange', 'ecolor': 'tab:orange', 'alpha': 0.45},
    {'color': 'tab:blue'}, {'color': 'tab:orange'},
    {'color': 'green'}, {'color': 'red'}, {'color': 'blue'},
    {'color': 'green'}, {'color': 'red'},
    {'color': 'green'}, {'color': 'tab:blue'},
    {'color': 'red'}, {'color': 'orange'},
]

ipfn_plots = plotting.ipfn_plots
fig, axes = ipfn_plots(
    x=x_data,
    y=y_data,
    ey=ey_data,
    boxes=boxes,
    nrows=3,
    ncols=1,
    order='column',
    labels=labels,
    linestyles=linestyles,
    markers=markers,
    markersizes=markersizes,
    plot_kw=plot_kw,
    x_labels=[None, None, PROFILE_RADIUS_LABEL],
    y_labels=['$T_e$', '$\\lambda_{Te}$', '$q_{||}$'],
    x_units_boxes=[None, None, 'm'],
    y_units_boxes=['eV', 'mm', 'GW/m$^2$'],
    x_mins_boxes=[r_zoom_min] * 3,
    x_maxs_boxes=[r_zoom_max] * 3,
    h_factor=1.3,
    v_factor=2.0,
    hspace=0.08,
    write_legends=True,
    title=f'Shot {shot_id} | {equilibrium} | {interval["t_begin"]:.3f}-{interval["t_end"]:.3f} s',
    show=False,
    close=False,
    save_fig=False,
)

axes = np.asarray(axes, dtype=object).reshape(-1)
for axis in axes[1:]:
    axis.sharex(axes[0])
for axis in axes:
    axis.set_xlim(r_zoom_min, r_zoom_max)
    for label, position, color in (
        ('EQH separatrix', physics_result['eqh']['Rsep_eqh'], 'black'),
        ('Spline separatrix', spline_physics['Rsep'], 'green'),
        ('mTanh separatrix', mtanh_physics['Rsep'], 'red'),
    ):
        axis.axvline(
            position, color=color, linestyle='--', linewidth=0.8,
            label=label if axis is axes[0] else None,
        )

eqh_rsep = float(physics_result['eqh']['Rsep_eqh'])
lambda_at_eqh = [
    np.interp(eqh_rsep, spline_r_physics, spline_lambda),
    np.interp(eqh_rsep, mtanh_r_physics, mtanh_lambda),
]
lambda_upper = np.nanmax(lambda_at_eqh)
lambda_upper += 0.05 * max(abs(lambda_upper), 1e-3)
axes[1].set_ylim(0, 60)
axes[2].set_ylim(0, 0.6)

for axis in axes[:-1]:
    axis.tick_params(axis='x', labelbottom=False)
axes[-1].set_xlabel(PROFILE_RADIUS_LABEL)
axes[0].legend(loc='upper right', ncol=2, fontsize=10)
from ipfnpytools.save_figure import save_figure
save_figure(fig,
             directory='/shares/users/work/joanune/Desktop/TESE_Joao_Pedro_Nunes/chapters/Formulation_and_Methods/Separatrix_Physics/figures',
             format='pdf', name =make_fig_name(shot_id, equilibrium, knots.size, interval, 'profiles_and_qs', spline_strategy=new_fit_config['fits']['spline']['knot_strategy'],
))
fig

In [ ]:
plots[]

In [ ]:
outliers_by_reason.keys()

In [ ]:
outliers_by_reason[outliers_by_reason['r_midplane']>2.22]

In [ ]:
def plot_strategy_profile(strategy_name, quantity, limits=None, show_fit=False, suffix='full'):
    strategy_data = profile_runs[strategy_name]
    result = strategy_data['result']
    mapped = result['_mapped']
    rejected = strategy_data['rejected']
    core = mapped.loc[mapped['system'].eq('core')]
    edge = mapped.loc[mapped['system'].eq('edge')]
    core_out = rejected.loc[rejected['system'].eq('core')]
    edge_out = rejected.loc[rejected['system'].eq('edge')]
    limits = limits or {}
    plot_outliers = {}

    if not core_out.empty:
        plot_outliers.update({
            'x_core_out': core_out['r_midplane'].to_numpy(),
            'y_core_out': core_out[quantity].to_numpy(),
            'y_err_core_out': core_out[f'{quantity}_err'].to_numpy(),
        })
    if not edge_out.empty:
        plot_outliers.update({
            'x_edge_out': edge_out['r_midplane'].to_numpy(),
            'y_edge_out': edge_out[quantity].to_numpy(),
            'y_err_edge_out': edge_out[f'{quantity}_err'].to_numpy(),
        })

    parameters = result['mtanh'][quantity]['params']
    branch_slug = strategy_name.lower().replace(' ', '_').replace('-', '_')
    y_label = {'Te': 'Te [eV]', 'Ne': 'Ne [m^-3]'}[quantity]
    return plot_profile_fit(
        x_core=core['r_midplane'].to_numpy(),
        x_edge=edge['r_midplane'].to_numpy(),
        y_core=core[quantity].to_numpy(),
        y_edge=edge[quantity].to_numpy(),
        y_err_core=core[f'{quantity}_err'].to_numpy(),
        y_err_edge=edge[f'{quantity}_err'].to_numpy(),
        time_c=core['time'].to_numpy(),
        time_e=edge['time'].to_numpy(),
        x_labels=PROFILE_RADIUS_LABEL,
        y_labels=y_label,
        popt=parameters,
        show_fit=show_fit,
        x_mins=limits.get('x_mins'),
        x_maxs=limits.get('x_maxs'),
        y_mins=limits.get('y_mins'),
        y_maxs=limits.get('y_maxs'),
        nshot=shot_id,
        interval=interval,
        equilibrium=equilibrium,
        fig_name=f'{shot_id}_{branch_slug}_{quantity}_{suffix}',
        directory=str(output_directory),
        save_fig=save_figures,
        show=show_figures,
        close=close_figures,
        **plot_outliers,
    )

## Full Profiles

Retained measurements, rejected-point overlays, and modified-tanh fits for both strategies. Legends are supplied by `plot_profile_fit`.

In [ ]:
for strategy_name in profile_runs:
    for quantity in ('Te', 'Ne'):
        plot_strategy_profile(
            strategy_name,
            quantity,
            show_fit=True,
            suffix='full',
            limits = {'y_maxs': 4000 if quantity == 'Te' else 1e20, 'y_mins': 0}
        )

## Zoom Profiles

Edit the axis-limit values below for each quantity. Set `zoom_show_fit` to `False` for data-only views or `True` to include the fit curve.

In [ ]:
zoom_axis_limits = {
    'Te': {
        'x_mins': 2.10,
        'x_maxs': 2.22,
        'y_mins': -100,
        'y_maxs': 1000,
    },
    'Ne': {
        'x_mins': 2.10,
        'x_maxs': 2.22,
        'y_mins': 0,
        'y_maxs': 5e19,
    },
}
zoom_show_fit = False

for strategy_name in profile_runs:
    for quantity in ('Te', 'Ne'):
        plot_strategy_profile(
            strategy_name,
            quantity,
            limits=zoom_axis_limits[quantity],
            show_fit=zoom_show_fit,
            suffix='zoom',
        )